In [1]:
import pydata_google_auth
from google.cloud import bigquery

credentials = pydata_google_auth.get_user_credentials(
    ["https://www.googleapis.com/auth/bigquery"],
)
client = bigquery.Client(project="off-nutrition-pipeline", credentials=credentials)

table = client.get_table("off-nutrition-pipeline.bronze.produits")
print(table.num_rows, "lignes")

C:\Users\ayoub\OneDrive\Bureau\.venv\lib\site-packages\google\api_core\_python_version_support.py:212: FutureWarning: You are using a non-supported Python version (3.10.5). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.api_core.
  warnings.warn(message, FutureWarning)
C:\Users\ayoub\OneDrive\Bureau\.venv\lib\site-packages\google\api_core\_python_version_support.py:212: FutureWarning: You are using a non-supported Python version (3.10.5). Google will not post any further updates to google.cloud.bigquery supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.cloud.bigquery.
  warnings.warn(message, FutureWarning)


4882161 lignes


In [2]:
for champ in table.schema:
    print(champ.name, "|", champ.field_type, "|", champ.mode)

code | STRING | NULLABLE
product_name | RECORD | NULLABLE
brands | STRING | NULLABLE
brands_tags | RECORD | NULLABLE
categories_tags | RECORD | NULLABLE
countries_tags | RECORD | NULLABLE
nutriscore_grade | STRING | NULLABLE
nutriscore_score | INTEGER | NULLABLE
no_nutrition_data | BOOLEAN | NULLABLE
nutrition_data_per | STRING | NULLABLE
nutriments_json | BYTES | NULLABLE


In [3]:
sql = """
SELECT
  COUNT(*) AS total,
  COUNTIF(brands IS NULL OR TRIM(brands) = '') AS sans_marque,
  COUNTIF(nutriscore_grade IS NULL) AS grade_null,
  COUNTIF(nutriscore_grade = 'unknown') AS grade_unknown,
  COUNTIF(nutriscore_score IS NULL) AS score_null,
  COUNTIF(no_nutrition_data) AS sans_donnees_nutrition,
  COUNTIF(IFNULL(ARRAY_LENGTH(categories_tags.list), 0) = 0) AS sans_categorie,
  COUNTIF(EXISTS(SELECT 1 FROM UNNEST(categories_tags.list) t WHERE t.element = 'en:null')) AS categorie_en_null,
  COUNTIF(EXISTS(SELECT 1 FROM UNNEST(countries_tags.list) t WHERE t.element = 'en:world')) AS pays_en_world
FROM `off-nutrition-pipeline.bronze.produits`
"""
ligne = list(client.query(sql).result())[0]
for nom, valeur in ligne.items():
    print(nom, ":", valeur)

total : 4882161
sans_marque : 1753485
grade_null : 66506
grade_unknown : 3273668
score_null : 3449795
sans_donnees_nutrition : 5238
sans_categorie : 2920591
categorie_en_null : 1189
pays_en_world : 73290


In [4]:
sql = """
SELECT nutriscore_grade, COUNT(*) AS nb
FROM `off-nutrition-pipeline.bronze.produits`
GROUP BY nutriscore_grade
ORDER BY nb DESC
"""
for ligne in client.query(sql).result():
    print(ligne["nutriscore_grade"], ":", ligne["nb"])

unknown : 3273668
e : 400170
d : 358554
c : 299411
a : 209342
b : 164889
not-applicable : 109621
None : 66506


In [5]:
sql = """
SELECT
  COUNT(*) AS total,
  COUNT(DISTINCT code) AS codes_distincts,
  COUNTIF(code IS NULL OR TRIM(code) = '') AS code_vide,
  COUNTIF(IFNULL(ARRAY_LENGTH(product_name.list), 0) = 0) AS sans_nom
FROM `off-nutrition-pipeline.bronze.produits`
"""
ligne = list(client.query(sql).result())[0]
for nom, valeur in ligne.items():
    print(nom, ":", valeur)

total : 4882161
codes_distincts : 4882100
code_vide : 1
sans_nom : 298361


In [6]:
sql = """
SELECT code, COUNT(*) AS nb
FROM `off-nutrition-pipeline.bronze.produits`
GROUP BY code
HAVING COUNT(*) > 1
ORDER BY nb DESC
LIMIT 15
"""
for ligne in client.query(sql).result():
    print(repr(ligne["code"]), ":", ligne["nb"])

'2700963001059' : 2
'30383354190402' : 2
'516408508322002749' : 2
'5304067005431' : 2
'3700439700348' : 2
'2000000151669' : 2
'8008082903405' : 2
'7340011495437' : 2
'0059527401555' : 2
'0059527601559' : 2
'31843340000818' : 2
'6449027002163' : 2
'809507300522001927' : 2
'8026380707232' : 2
'2000000151670' : 2


In [7]:
sql = """
SELECT
  code,
  brands,
  product_name.list[SAFE_OFFSET(0)].element.text AS nom,
  nutriscore_grade,
  nutriscore_score,
  nutrition_data_per
FROM `off-nutrition-pipeline.bronze.produits`
WHERE code IN ('2700963001059', '3700439700348', '8008082903405')
ORDER BY code
"""
for ligne in client.query(sql).result():
    print(dict(ligne.items()))

{'code': '2700963001059', 'brands': 'Palais des Thés', 'nom': 'Jardin anglais', 'nutriscore_grade': 'unknown', 'nutriscore_score': None, 'nutrition_data_per': None}
{'code': '2700963001059', 'brands': '', 'nom': 'Jardin anglais', 'nutriscore_grade': None, 'nutriscore_score': None, 'nutrition_data_per': '100g'}
{'code': '3700439700348', 'brands': '', 'nom': None, 'nutriscore_grade': 'unknown', 'nutriscore_score': None, 'nutrition_data_per': '100g'}
{'code': '3700439700348', 'brands': '', 'nom': None, 'nutriscore_grade': None, 'nutriscore_score': None, 'nutrition_data_per': '100g'}
{'code': '8008082903405', 'brands': 'Cucina nostrana', 'nom': 'Melanzane alla parmigiana', 'nutriscore_grade': 'b', 'nutriscore_score': 1, 'nutrition_data_per': '100g'}
{'code': '8008082903405', 'brands': 'Cucina Nostrana', 'nom': 'Melanzane alla parmigiana', 'nutriscore_grade': 'b', 'nutriscore_score': 1, 'nutrition_data_per': None}


In [8]:
dataset = bigquery.Dataset("off-nutrition-pipeline.silver")
dataset.location = "EU"
client.create_dataset(dataset, exists_ok=True)
print("silver prête")

silver prête
